# 🧠 16 · Deep Learning for Bioimage Classification
### From hand-crafted features to learned ones: neurons, backpropagation, and a small CNN

**Part D · Classifying bioimages** · Module 16 of 37 (core) · builds on 15

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/16_deep_learning_classification.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (a GPU runtime speeds up Section 7, but is not required) |
| **Core frameworks** | NumPy, PyTorch |
| **Prerequisites** | Module 15 (classical classification) |
| **Estimated time** | 100–120 minutes |
| **02-740 topics covered** | artificial neurons, forward propagation, activation functions, the softmax classifier, loss functions, gradient descent and optimizers, backpropagation, CNN layers (convolution/pooling/FC/dropout/batch norm), transfer learning, class imbalance in bioimage datasets |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All training images below are generated on the fly by `course_toolkit.simulate_nuclei_image`; no image, weight file, or dataset is stored anywhere in this notebook.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain forward propagation through an artificial neuron and a small network, and compare the linear,
  sigmoid, tanh and ReLU activation functions.
- Derive the softmax classifier and the cross-entropy loss it pairs with, and compute both by hand on a
  toy example.
- Explain gradient descent, and name what stochastic gradient descent, momentum, and Adam each add to it.
- Work through backpropagation on a small computational graph by hand, matching an automatic-differentiation
  result.
- Describe the role of convolutional, pooling, fully-connected, dropout and batch-normalization layers in
  a CNN, and train a small CNN in PyTorch to classify simulated nuclei by shape.
- Explain why class imbalance hurts bioimage classifiers in practice, and how transfer learning lets a
  model trained on one task adapt quickly to a related one with little new data.

## 🗺️ Contents

1. From features to learned features
2. Artificial neurons and forward propagation
3. Activation functions
4. The softmax classifier and cross-entropy loss
5. Gradient descent and optimizers
6. Backpropagation, worked by hand
7. Convolutional neural networks: the building blocks
8. Training a small CNN on simulated nuclei
9. Class imbalance
10. Transfer learning

## 🧫 The biological question

Module 15's hand-crafted features (GLCM texture, compactness, aspect ratio...) worked, but someone had to
*think of* each one. A deep network instead **learns its own features** directly from pixels, discovering
whatever combination of edges, textures and shapes best separates the classes — at the cost of needing
more data and being harder to inspect. This trade-off is exactly why bioimage classification papers from
the 2010s onward (HEp-2 cell classification, blood cell typing, subcellular-pattern classification) moved
from the Module 15 pipeline to convolutional neural networks, while keeping the same underlying biological
questions.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(80)
SEED = 80
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1. From features to learned features

Deep learning is, in one sentence, a class of machine learning algorithms that uses **multiple layers** to
progressively build higher-level features directly from raw input — rather than being handed hand-crafted
features like Module 15's GLCM contrast or compactness. Bioimage datasets are a particularly good fit for
this idea and a particularly hard case for it at the same time: they are often high-volume and
high-throughput (deep learning thrives on data), but also low signal-to-noise and highly variable across
instruments and labs (deep learning can overfit to quirks of one dataset). The rest of this notebook builds
up, piece by piece, the network we will eventually train: neurons → activations → a softmax output → a loss
→ an optimizer → backpropagation to compute gradients → convolutional layers to exploit image structure.

## 2. Artificial neurons and forward propagation

An artificial neuron takes a weighted sum of its inputs plus a bias, then applies a non-linear
**activation function** $f$:

$$o_j = f\Big(\sum_i w_{j,i}\, o_{j-1,i} + b_j\Big)$$

Stacking neurons into layers, and layers into a network, gives **forward propagation**: inputs flow layer
by layer to an output. The simplest network (one hidden layer, fully connected, no convolution) is often
called a multilayer perceptron (MLP) or vanilla deep neural network (DNN).

In [ ]:
# ── 2.1 Forward propagation through a tiny 2-layer network, by hand ──
def relu(z):
    return np.maximum(0, z)

# inputs: 3 toy "features" of one simulated object
x = np.array([0.6, -0.2, 1.1])
W1 = np.array([[0.3, -0.5, 0.8], [0.1, 0.4, -0.2]])   # 2 hidden neurons
b1 = np.array([0.05, -0.1])
W2 = np.array([[0.7, -0.3]])                           # 1 output neuron
b2 = np.array([0.0])

h = relu(W1 @ x + b1)      # hidden layer activations
out = W2 @ h + b2          # linear output (no activation yet)
print("hidden activations :", h)
print("raw output score   :", out)

🔍 **What just happened?** Each hidden neuron computed its own weighted sum
of the three input features, applied ReLU, and the output neuron combined both hidden activations. This
*is* forward propagation — the only difference from a full CNN is scale (millions of neurons instead of
three) and structure (convolutional weight sharing instead of one weight per pixel, Section 7).

## 3. Activation functions

| Activation | Formula | Range | Note |
|---|---|---|---|
| Linear | $f(z)=z$ | $(-\infty,\infty)$ | no non-linearity — stacking linear layers collapses to one |
| Sigmoid | $f(z)=1/(1+e^{-z})$ | $(0,1)$ | saturates for $\lvert z\rvert>3$: gradients vanish there |
| Tanh | $f(z)=\tanh(z)$ | $(-1,1)$ | zero-centered version of sigmoid; still saturates |
| ReLU | $f(z)=\max(0,z)$ | $[0,\infty)$ | the default in modern CNNs: cheap, non-saturating for $z>0$ |

*In words:* sigmoid and tanh are very sensitive to inputs near zero but barely respond to large-magnitude
inputs (their gradient flattens out there, which slows or stalls learning in deep stacks — the
"vanishing gradient" problem); ReLU instead stays linear for all positive inputs, which is why it largely
replaced sigmoid/tanh as the default hidden-layer activation once networks got deep.

In [ ]:
# ── 3.1 Plot the four activation functions ──
z = np.linspace(-6, 6, 300)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(z, z, label="linear")
ax.plot(z, 1 / (1 + np.exp(-z)), label="sigmoid")
ax.plot(z, np.tanh(z), label="tanh")
ax.plot(z, np.maximum(0, z), label="ReLU")
ax.set_ylim(-2, 3); ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
ax.legend(); ax.set_xlabel("z"); ax.set_ylabel("f(z)"); ax.set_title("Activation functions")
plt.show()

✏️ **Your turn.** At $z=5$, compute the *slope* (derivative) of sigmoid
($f(z)(1-f(z))$), tanh ($1-\tanh^2(z)$), and ReLU (1 for $z>0$). Which activation has effectively stopped
learning anything from further increases in $z$, and which keeps a constant, healthy gradient?

## 4. The softmax classifier and cross-entropy loss

A classification network's final layer outputs one raw score per class. The **softmax** function turns
these scores $s_1,\dots,s_C$ into a valid probability distribution:

$$p_c = \frac{e^{s_c}}{\sum_{k=1}^{C} e^{s_k}}, \qquad p_c \ge 0, \quad \sum_c p_c = 1$$

Given the true class $c^\star$, the **cross-entropy loss** is $L = -\log p_{c^\star}$: zero when the model
assigns probability 1 to the correct class, and growing as that probability shrinks toward zero. Averaged
over a batch of examples, this becomes the training loss that gradient descent (Section 5) minimizes.
Cross-entropy is mathematically identical to maximizing the likelihood of the true labels (maximum
likelihood estimation): minimizing $-\log p_{c^\star}$ *is* maximizing $p_{c^\star}$.

In [ ]:
# ── 4.1 Softmax and cross-entropy on three toy class scores ──
scores = np.array([3.2, -1.7, 5.1])          # raw scores for, say, "round", "ring", "dividing"
exp_scores = np.exp(scores)
probs = exp_scores / exp_scores.sum()
true_class = 2   # suppose the true label is "dividing" (index 2)
loss = -np.log(probs[true_class])
print("exp(scores)   :", np.round(exp_scores, 2))
print("probabilities :", np.round(probs, 3), " (sums to", round(probs.sum(), 3), ")")
print(f"cross-entropy loss for the true class: {loss:.3f}")

true_class_wrong = 1  # what if the true label had instead been "ring" (index 1, the LOWEST score)?
print(f"cross-entropy loss if the true class were index 1 instead: {-np.log(probs[true_class_wrong]):.3f}")

🧪 **Ground-truth check.** The loss is small (near 0) when the softmax already
puts most of its probability mass on the correct class, and large when it does not — confirm this by
comparing the two losses printed above: assigning the *true* label to the class with the *highest* score
should give a much smaller loss than assigning it to the class with the lowest score.

## 5. Gradient descent and optimizers

Training means searching over the millions of weights for the setting that minimizes the average loss —
visualized as finding the lowest point of a very high-dimensional "loss landscape." **Gradient descent**
repeatedly takes a step in the direction that decreases loss fastest (the negative gradient):
$\theta \leftarrow \theta - \eta\,\nabla_\theta J(\theta)$, where $\eta$ is the **learning rate**.
Computing the gradient over the *entire* training set every step is expensive, so **stochastic gradient
descent (SGD)** instead estimates it from one small random *mini-batch* at a time. Plain SGD can oscillate
in narrow valleys of the loss landscape; **momentum** smooths this out by accumulating a running average of
past gradients; **Adam** goes further, adaptively rescaling the learning rate per-parameter from running
estimates of both the gradient and its square. Random weight search (try random weights, keep the best) was
an early idea but is far too slow for any network with more than a handful of parameters.

In [ ]:
# ── 5.1 Gradient descent on a simple 2-D bowl-shaped loss, three step sizes ──
def loss_fn(w):
    return 0.5 * w[0] ** 2 + 2.0 * w[1] ** 2   # a steep, elongated "ravine"

def grad_fn(w):
    return np.array([w[0], 4.0 * w[1]])

fig, ax = plt.subplots(figsize=(5, 4))
gw1, gw2 = np.meshgrid(np.linspace(-3, 3, 60), np.linspace(-3, 3, 60))
ax.contour(gw1, gw2, 0.5 * gw1 ** 2 + 2.0 * gw2 ** 2, levels=12, cmap="Greys")
for lr, color in [(0.1, "tab:blue"), (0.4, "tab:orange"), (0.55, "tab:red")]:
    w = np.array([2.6, 2.6])
    path = [w.copy()]
    for _ in range(25):
        w = w - lr * grad_fn(w)
        path.append(w.copy())
    path = np.array(path)
    ax.plot(path[:, 0], path[:, 1], "o-", color=color, markersize=3, label=f"learning rate={lr}")
ax.legend(); ax.set_title("Gradient descent: step size matters"); plt.show()

🔍 **What just happened?** The small learning rate converges smoothly but
slowly; the medium one converges fast; the large one overshoots and oscillates (and, for a bowl this steep,
a slightly larger step size still would diverge outright). This is exactly why adaptive optimizers like
Adam exist — to get fast convergence *without* having to hand-tune a single fixed learning rate that
works for every parameter and every stage of training.

## 6. Backpropagation, worked by hand

Backpropagation computes the gradient of the loss with respect to *every* weight in the network using the
chain rule, working backward from the output. For a tiny computational graph $q = x + y,\; f = q \cdot z$
with $x=-5, y=2, z=-4$:

- Forward pass: $q = x+y = -3$, $f = q\cdot z = 12$.
- $\partial f/\partial f = 1$ (the gradient of the output with respect to itself).
- $\partial f/\partial q = z = -4$, and $\partial f/\partial z = q = -3$.
- Chain rule: $\partial f/\partial x = \partial f/\partial q \cdot \partial q/\partial x = -4 \cdot 1 = -4$,
  and likewise $\partial f/\partial y = -4$.

The same chain rule, applied automatically to millions of nodes, is what lets a CNN's weights update after
every mini-batch.

In [ ]:
# ── 6.1 Reproduce the hand-derivation with PyTorch autograd ──
x = torch.tensor(-5.0, requires_grad=True)
y = torch.tensor(2.0, requires_grad=True)
z = torch.tensor(-4.0, requires_grad=True)
q = x + y
f = q * z
f.backward()
print(f"f = {f.item():.1f}  (hand-derivation: 12)")
print(f"df/dx = {x.grad.item():.1f}  df/dy = {y.grad.item():.1f}  df/dz = {z.grad.item():.1f}"
      "  (hand-derivation: -4, -4, -3)")

✏️ **Your turn.** Change `z` to `-1.0` and re-run. Work out by hand what
`df/dx` *should* become (hint: it is still $z$, just a different value) before checking against autograd's
answer. This is precisely the mechanism — chain rule, applied node by node — that trains every network in
this notebook; PyTorch's `autograd` just automates bookkeeping that would be impractical to do by hand for
a network with millions of weights.

### 📘 Lecture notes — vectorized backpropagation

For a layer $y = Wx$ with loss $L$ depending on $y$, the chain rule gives
$\partial L/\partial W_{i,j} = \partial L/\partial y_i \cdot x_j$ and
$\partial L/\partial x_j = \sum_i \partial L/\partial y_i \cdot W_{i,j}$ — in matrix form,
$\partial L/\partial W = (\partial L/\partial y)\, x^\top$ and
$\partial L/\partial x = W^\top (\partial L/\partial y)$. This is exactly the scalar chain rule from
above, just applied to every entry of a vector or matrix at once; it is what every deep learning framework
implements under the hood for every layer type, including the convolutional layers in Section 7.

## 7. Convolutional neural networks: the building blocks

A CNN replaces the fully-connected layers of an MLP, for at least its early layers, with layers that
exploit images' spatial structure:

- **Convolutional layer** — like Module 11's fixed filters (Sobel, Gaussian), but now the kernel's weights
  are *learned* from data rather than hand-designed. Every patch of the image is processed by the *same*
  kernel ("weight sharing"), which is both far more parameter-efficient than a fully-connected layer and a
  good match for images, where a useful local pattern (an edge, a blob) can appear anywhere.
- **Pooling layer** (max or average) — shrinks each feature map by summarizing small neighbourhoods,
  giving the network some translation tolerance and reducing computation for later layers.
- **Fully-connected (FC) layer** — after several conv/pool stages, the remaining feature maps are
  flattened into a vector and passed through one or more FC layers (identical to the MLP layers from
  Section 2) ending in the softmax classifier from Section 4.
- **Dropout** — randomly zeroes a fraction of activations during training only, which prevents neurons
  from co-adapting too tightly to each other and acts as a strong, cheap regularizer against overfitting.
- **Batch normalization** — rescales each layer's inputs to have a stable mean and variance within every
  mini-batch, counteracting "internal covariate shift" (earlier layers' changing outputs constantly
  shifting what later layers see) and allowing higher learning rates.

In [ ]:
# ── 7.1 A learned convolution kernel vs. a hand-designed one (Module 11/14) ──
sample = simulate_nuclei_image(n_cells=1, size=48, shape_classes=("elongated",), seed=3)["image"]
hand_kernel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)     # Sobel-x (hand-designed)
learned_kernel = rng.normal(0, 0.3, size=(3, 3))                              # what a CNN starts from
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
show_image(axes[0], sample, "input image")
show_image(axes[1], ndi.correlate(sample, hand_kernel), "hand-designed (Sobel-x)")
show_image(axes[2], ndi.correlate(sample, learned_kernel), "randomly-initialized 'learned' kernel")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both panels on the right are literally the same
operation (a 3×3 convolution) applied to the same image — the *only* difference is where the nine numbers
in the kernel came from. Training a CNN is the process of nudging kernels that start out as meaningless
noise (right panel) into ones that extract useful structure (middle panel), using backpropagation to figure
out which direction to nudge each of the nine numbers.

## 8. Training a small CNN on simulated nuclei

We now put it all together: generate single-cell crops across all six `SHAPE_CLASSES`, build a small CNN
(two conv+pool blocks, dropout, one FC layer, softmax via `nn.CrossEntropyLoss`), and train it with Adam.

In [ ]:
# ── 8.1 Build a labelled dataset of single-shape crops ──
CLASS_NAMES = list(SHAPE_CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

def make_dataset(n_per_class, size=32, seed0=0):
    X, y = [], []
    for c in CLASS_NAMES:
        for k in range(n_per_class):
            sim = simulate_nuclei_image(n_cells=1, size=size, shape_classes=(c,),
                                         noise_sigma=rng.uniform(0.02, 0.05), seed=seed0 + k)
            X.append(sim["image"])
            y.append(CLASS_TO_IDX[c])
    X = np.stack(X)[:, None, :, :].astype(np.float32)   # (N, 1, H, W)
    y = np.array(y, dtype=np.int64)
    order = rng.permutation(len(y))
    return torch.from_numpy(X[order]), torch.from_numpy(y[order])

X_train, y_train = make_dataset(60, seed0=2000)
X_val, y_val = make_dataset(15, seed0=5000)
print("train set:", X_train.shape, " val set:", X_val.shape)

In [ ]:
# ── 8.2 A small CNN: conv -> pool -> conv -> pool -> dropout -> FC -> softmax (via CE loss) ──
class SmallCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(16 * 8 * 8, n_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.bn1(self.conv1(x))), 2)   # 32x32 -> 16x16
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)              # 16x16 -> 8x8
        x = self.dropout(x.flatten(1))
        return self.fc(x)   # raw scores; nn.CrossEntropyLoss applies softmax internally

model = SmallCNN(n_classes=len(CLASS_NAMES))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
print(model)

In [ ]:
# ── 8.3 Train for a few epochs (small data, small model: seconds on CPU) ──
history = []
for epoch in range(20):
    model.train()
    perm = torch.randperm(len(X_train))
    epoch_loss = 0.0
    for i in range(0, len(X_train), 32):
        idx = perm[i:i + 32]
        optimizer.zero_grad()
        out = model(X_train[idx])
        loss = criterion(out, y_train[idx])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    model.eval()
    with torch.no_grad():
        val_acc = (model(X_val).argmax(1) == y_val).float().mean().item()
    history.append((epoch_loss / len(X_train), val_acc))

for e in (0, 4, 9, 14, 19):
    tl, va = history[e]
    print(f"epoch {e+1:2d}:  train loss={tl:.3f}  val accuracy={va:.2f}")

In [ ]:
# ── 8.4 Plot the training curve ──
losses, accs = zip(*history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))
ax1.plot(losses); ax1.set_title("training loss"); ax1.set_xlabel("epoch")
ax2.plot(accs); ax2.set_title("validation accuracy"); ax2.set_xlabel("epoch"); ax2.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** Because `simulate_nuclei_image` hands back the
exact shape label used to draw each crop, `val_acc` above is computed against a perfectly reliable ground
truth — something a real bioimage project only gets via careful (and expensive) manual or
experimentally-designed annotation.

✏️ **Your turn.** Set `self.dropout = nn.Dropout(0.0)` (i.e., disable
dropout) and re-run Sections 8.2–8.4. With a training set this small, does validation accuracy become more
or less stable across epochs? This is dropout's regularizing effect — "preventing complex co-adaptations
on training data" — made concrete.

## 9. Class imbalance

Real bioimage datasets are rarely balanced: a HEp-2 cell classification study, for instance, has to work
with however many examples of each staining pattern happened to be captured and annotated, and some
patterns are simply rarer than others. A classifier trained on an imbalanced dataset can reach high
*overall* accuracy by essentially always predicting the majority class — exactly the failure mode we
should watch for.

In [ ]:
# ── 9.1 Build a deliberately imbalanced training set and retrain ──
def make_imbalanced_dataset(counts, size=32, seed0=9000):
    X, y = [], []
    for c, n in counts.items():
        for k in range(n):
            sim = simulate_nuclei_image(n_cells=1, size=size, shape_classes=(c,),
                                         noise_sigma=rng.uniform(0.02, 0.05), seed=seed0 + k)
            X.append(sim["image"]); y.append(CLASS_TO_IDX[c])
    X = np.stack(X)[:, None, :, :].astype(np.float32)
    y = np.array(y, dtype=np.int64)
    order = rng.permutation(len(y))
    return torch.from_numpy(X[order]), torch.from_numpy(y[order])

counts = {"round": 90, "dividing": 90, "elongated": 6, "kidney": 6, "ring": 6, "irregular": 6}
X_imb, y_imb = make_imbalanced_dataset(counts)
model_imb = SmallCNN(n_classes=len(CLASS_NAMES))
opt_imb = torch.optim.Adam(model_imb.parameters(), lr=1e-3)
for epoch in range(20):
    model_imb.train()
    perm = torch.randperm(len(X_imb))
    for i in range(0, len(X_imb), 32):
        idx = perm[i:i + 32]
        opt_imb.zero_grad()
        loss = criterion(model_imb(X_imb[idx]), y_imb[idx])
        loss.backward(); opt_imb.step()

model_imb.eval()
with torch.no_grad():
    pred = model_imb(X_val).argmax(1)
overall_acc = (pred == y_val).float().mean().item()
print(f"overall validation accuracy: {overall_acc:.2f}")
for c in CLASS_NAMES:
    mask = y_val == CLASS_TO_IDX[c]
    per_class_acc = (pred[mask] == y_val[mask]).float().mean().item() if mask.any() else float("nan")
    print(f"  recall for {c:10s} (n_train={counts[c]:3d}): {per_class_acc:.2f}")

🔍 **What just happened?** Overall accuracy can look deceptively fine while
the rare classes (only 6 training examples each) are recalled far worse than the common ones (90 training
examples each) — exactly the pattern reported for real imbalanced bioimage datasets. Standard fixes include
oversampling rare classes, weighting the loss function by inverse class frequency, and — the subject of the
next section — transfer learning from a model already trained on related, more abundant data.

## 10. Transfer learning

**Transfer learning** reuses knowledge from one trained model to speed up learning on a new, related task
— typically by keeping (freezing) the early convolutional layers, which tend to learn fairly generic
edge/texture detectors, and only retraining the final layer(s) on the new task's (often much smaller)
labelled set. This is especially valuable in bioimage analysis, where annotated data for a *specific* new
experiment is often scarce even when a related, larger annotated dataset exists.

In [ ]:
# ── 10.1 Freeze the base model's convolutions; retrain only the FC head for new classes ──
# Pretend "round" and "dividing" were the original, data-rich task, and "ring" is a brand-new,
# data-poor class we must now also recognize.
base_classes = ["round", "dividing"]
new_classes = ["round", "dividing", "ring"]

X_base, y_base = make_dataset(80, size=32, seed0=1000)
mask_base = np.isin(y_base.numpy(), [CLASS_TO_IDX[c] for c in base_classes])
X_base, y_base_raw = X_base[mask_base], y_base[mask_base]
remap_base = {CLASS_TO_IDX[c]: i for i, c in enumerate(base_classes)}
y_base = torch.tensor([remap_base[v.item()] for v in y_base_raw])

base_model = SmallCNN(n_classes=len(base_classes))
opt_base = torch.optim.Adam(base_model.parameters(), lr=1e-3)
for epoch in range(15):
    perm = torch.randperm(len(X_base))
    for i in range(0, len(X_base), 32):
        idx = perm[i:i + 32]
        opt_base.zero_grad()
        loss = criterion(base_model(X_base[idx]), y_base[idx])
        loss.backward(); opt_base.step()
print("base model trained on:", base_classes)

In [ ]:
# ── 10.2 Compare: fine-tune only the FC head vs. train an identical model from scratch ──
def few_shot_set(classes, n_per_class=5, seed0=7000):
    X, y = [], []
    for c in classes:
        for k in range(n_per_class):
            sim = simulate_nuclei_image(n_cells=1, size=32, shape_classes=(c,), seed=seed0 + k)
            X.append(sim["image"]); y.append(classes.index(c))
    X = np.stack(X)[:, None, :, :].astype(np.float32)
    return torch.from_numpy(X), torch.tensor(y, dtype=torch.int64)

X_few, y_few = few_shot_set(new_classes, n_per_class=5)          # tiny: 5 examples per class
X_few_val, y_few_val = few_shot_set(new_classes, n_per_class=15, seed0=8000)

def retrain_head_only(base_model, n_new_classes, X, y, epochs=30):
    import copy
    tuned = copy.deepcopy(base_model)
    for p in tuned.conv1.parameters(): p.requires_grad = False
    for p in tuned.conv2.parameters(): p.requires_grad = False
    tuned.fc = nn.Linear(tuned.fc.in_features, n_new_classes)   # fresh head for the new class count
    opt = torch.optim.Adam(tuned.fc.parameters(), lr=1e-2)      # only the head's weights are trained
    for _ in range(epochs):
        opt.zero_grad(); loss = criterion(tuned(X), y); loss.backward(); opt.step()
    return tuned

transferred = retrain_head_only(base_model, len(new_classes), X_few, y_few)
from_scratch = SmallCNN(n_classes=len(new_classes))
opt_scratch = torch.optim.Adam(from_scratch.parameters(), lr=1e-2)
for _ in range(30):
    opt_scratch.zero_grad(); loss = criterion(from_scratch(X_few), y_few); loss.backward(); opt_scratch.step()

with torch.no_grad():
    acc_transfer = (transferred(X_few_val).argmax(1) == y_few_val).float().mean().item()
    acc_scratch = (from_scratch(X_few_val).argmax(1) == y_few_val).float().mean().item()
print(f"transfer learning (frozen convs, {5*len(new_classes)} new examples): {acc_transfer:.2f}")
print(f"trained from scratch (same {5*len(new_classes)} new examples)      : {acc_scratch:.2f}")

✏️ **Your turn.** With only 5 examples per class, "from scratch" has to learn
*everything* — including low-level edge/blob detectors that are mostly the same for every shape — from
almost no data, while the transferred model only has to learn how to combine *already-useful* features for
one new class. Increase `n_per_class` in `few_shot_set` to 20 and re-run: does the gap between the two
strategies shrink, as you would expect once "from scratch" finally has enough data of its own?

📌 **Key takeaways**

- A CNN learns its own features end-to-end via gradient descent, in contrast to Module 15's hand-crafted texture/shape features — at the cost of needing more data and more compute, and being harder to interpret directly.
- Softmax turns raw class scores into probabilities; cross-entropy loss, minimized by gradient descent, is the same objective as maximum-likelihood training.
- Backpropagation is just the chain rule, applied systematically backward through the computational graph, to get the gradient of the loss with respect to every weight.
- Convolution (weight sharing), pooling (spatial summarization), dropout (regularization) and batch normalization (stabilizing layer inputs) are the standard CNN building blocks.
- Class imbalance can hide badly in an overall accuracy number; always check per-class recall. Transfer learning — freezing early, generic layers and only retraining the final layer(s) — is a standard, effective remedy when new, related data is scarce.

✅ **Checkpoint**

1. Why does cross-entropy loss use $-\log p_{c^\star}$ rather than, say, $1 - p_{c^\star}$?
   <details><summary>Answer</summary>The log makes the loss blow up as $p_{c^\star} \to 0$ (heavily
   penalizing confident wrong answers) and makes the gradient well-behaved near $p_{c^\star}=1$; it also
   makes cross-entropy exactly equal to negative log-likelihood, connecting training to maximum
   likelihood estimation.</details>
2. Why does a convolutional layer need far fewer parameters than a fully-connected layer processing the
   same image?
   <details><summary>Answer</summary>Weight sharing: the same small kernel (e.g. 3×3×8 weights) is
   applied at every spatial location, instead of a fully-connected layer needing one independent weight
   per input-pixel/output-neuron pair.</details>
3. A model gets 95% overall validation accuracy on a 6-class problem where one class makes up 80% of the
   validation set. Is 95% necessarily good news?
   <details><summary>Answer</summary>No — a model that always predicts the majority class alone would
   score roughly 80%, so 95% overall could still hide very poor recall on the minority classes; always
   check a per-class breakdown (or a confusion matrix) under imbalance.</details>

🧠 **Questions & discussion**

- The HEp-2 classification study in the lecture material rotated its images to artificially expand the training set. Why does this kind of augmentation make particular sense for microscopy images, where a cell's orientation under the microscope is biologically arbitrary?
- We froze *all* of the convolutional layers during transfer learning in Section 10. In what situation might you instead want to freeze only the *earliest* layer and allow later convolutional layers to keep adapting?
- Batch normalization computes statistics over a mini-batch. What could go wrong if you tried to use a batch size of 1?

🚀 **Bonus — going further**

- Visualize what `conv1`'s eight learned 3×3 kernels look like after training (compare to the random kernel in Section 7.1) — do any resemble edge or blob detectors?
- Implement a simple learning-rate schedule (halve the learning rate every 10 epochs) and compare the resulting training curve to Section 8.4.
- Add weighted loss (`nn.CrossEntropyLoss(weight=...)`, inversely proportional to class frequency) to the imbalanced-dataset experiment in Section 9 and see whether per-class recall becomes more even.

📝 **Lab note**

```text
Module 16 — Deep learning classification
Date: ____________
CNN architecture: conv(___) -> pool -> conv(___) -> pool -> dropout(___) -> FC
Balanced validation accuracy: ____  Imbalanced per-class recall (best/worst): ____ / ____
Transfer-learning accuracy vs. from-scratch (same few-shot data): ____ vs. ____
One hyperparameter I would tune next, and why: ____________
```

📚 **References**

- LeCun, Y., Bottou, L., Bengio, Y. & Haffner, P. (1998). Gradient-based learning applied to document recognition. *Proceedings of the IEEE*, 86(11), 2278–2324.
- Kingma, D. P. & Ba, J. (2015). Adam: A method for stochastic optimization. *ICLR 2015*.
- Ioffe, S. & Szegedy, C. (2015). Batch normalization: Accelerating deep network training by reducing internal covariate shift. *ICML 2015*.
- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. & Salakhutdinov, R. (2014). Dropout: A simple way to prevent neural networks from overfitting. *JMLR*, 15(1), 1929–1958.
- Gao, Z., Wang, L., Zhou, L. & Zhang, R. (2017). HEp-2 cell image classification with deep convolutional neural networks. *IEEE Journal of Biomedical and Health Informatics*, 21(2), 416–428.
- Boland, M. V. & Murphy, R. F. (2001). A neural network classifier capable of recognizing the patterns of all major subcellular structures in fluorescence microscope images of HeLa cells. *Bioinformatics*, 17(12), 1213–1223.
- Yosinski, J., Clune, J., Bengio, Y. & Lipson, H. (2014). How transferable are features in deep neural networks? *NeurIPS 2014*.

➡️ **Next.** Module 17 asks what happens once we have *many* features (or a whole learned representation) per object: how do we visualize and reduce that high-dimensional space down to something a biologist can actually look at? That is dimensionality reduction — PCA, MDS, Isomap, t-SNE, and autoencoders.